## Generate (anchor, positive) data from chunk files

In [1]:
import pyarrow
import os
import pandas as pd
import numpy as np
import hashlib

import ast
import json
import regex as re
from pprint import pprint

import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

from google.colab import userdata
from huggingface_hub import HfApi
from datasets import load_dataset, Dataset

In [ ]:
import os

if int(os.environ.get("COLAB_GPU", 0)) > 0:
    print("🚀 Running on a GPU environment!")
elif "COLAB_TPU_ADDR" in os.environ and os.environ["COLAB_TPU_ADDR"]:
    print("⚡ Running on a TPU environment!")
else:
    print("🖥️ Running on a standard CPU environment.")

ValueError: invalid literal for int() with base 10: ''

### **Note**:-<u>Going forward in this notebook, following references are *impicable by comments, texts containing old data(iter1 and 2) and new data(iter1 and 2)*</u>:-
- **Old data**:-the *4 synthetic anchors mapped to (generated from) 1 consolidated chunk/positive(title +summary+ inclusion criteria)* per trial record(unique nctId). Coarser but semantically strong one to one anchor-positive pair mapping.
    - It has 2 iterations(<u>iter1:- the oldest iterations</u>) and iter2(<u>the final iteration of data generation and modelling:-best_model</u>)
    - **<u>Data_preprocess iter1</u>**:-The iteration1 assumed 2 fold discrete stratification of inclusion-exclusion blocks insiand included simpler regex.
    - **<u>Data preprocess iter2</u>**:-
        - while iteration 2 also accounted for corner cases where inclusion-exclusion occur in muliple(>2) subchunks with 2block separation. Hence complex regex in layered fashion use to derive at clean chunks. 
        - Also chunk length normalized by excluding outlier margin at 90% confidence.
    
    - The [data](https://huggingface.co/datasets/vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data) for iter1 can be refrred by link. Simlarly is case for iter2(final version), [final data](https://huggingface.co/datasets/vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data_final) ---> all present on **huggingface hub**.

- **new data**:- the *5 synthetic anchors mapped to (generated from) 3 chunks /positive(title-->1, summary-->2, inclusion criteria-->2 individually)* per trial record(unique nctId). Finer and more directed, precise one to one anchor-positive pair mapping.
    - It has 2 iterations iteration1 and iter2.
    - **<u>Data preprocess iter 1 & 2</u>**:- 
        - In its both iterations all the procedure of **Data preprocess iter2** (layered complex regex + oultlier eradication(at 90% confidence) via density line-curve plot) were followed. 
        - Just an addition of **unique_chunk_identifier** was done across different trial records where granular chunks appeared same(only **inclusion criteria** seemed to repeat due to incremental nature of certain trial groups).
    - Two iterations done with differnce being orienation of relevant_documents. In 1st iteration, *row_id* and *unique_chunk_identifier* was done. In 2nd iteration, *row_id* and *document_id(nctId+block_no)* was done.
    - The [data](https://huggingface.co/datasets/vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data2) for iter1 can be refrred by link. The iter 2 dataset yeilded below par results so did't save those. 

In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
old_data_folder, new_data_folder ="/old_data_iter1", "/new_data_iter2"
path = "/content/gdrive/MyDrive/Embedding_model"

df_positive_chunk_old = pd.read_parquet(os.path.join(path+old_data_folder,"CT_id_+veCleanedChunks.parquet"))#"CT_id_+veChunks.parquet"
df_positive_chunk_new = pd.read_parquet(os.path.join(path+new_data_folder, "CT_id_+veGranulatedChunks.parquet"))

In [ ]:
df_positive_chunk_old.head()

,nctId,Positive_chunks,char_len,cluster_id
0,NCT07631078,TITLE: kTMP-Enhanced Motor Rehabilitation for ...,750,0
1,NCT07640360,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,0
2,NCT07635498,TITLE: Virtual Stroke Units Versus Conventiona...,2105,0
3,NCT07690982,TITLE: Validity and Reliability of the PhysioM...,1839,0
4,NCT07702968,TITLE: Validation of Low Field MRI in Patients...,1296,0


In [ ]:
df_positive_chunk_new.head()

,nctId,chunk,chunk_length,type,block_no
0,NCT07661732,eHealth Lifestyle Intervention To Enhance Outc...,198,title,1
1,NCT07661732,The goal of this clinical trial is to learn if...,1202,summary,2
2,NCT07661732,* Age over 18 years\n* Body mass index (BMI) o...,501,inclusion_criteria,3
3,NCT07704034,Wearable-Triggered Digital Safety Net for Real...,108,title,1
4,NCT07704034,This research study is for participants diagno...,515,summary,2


reading imporatnt elements from schema(system_prompt, anchorlist_scehma, anchor_schema, empty_results_schema)

In [ ]:
def readfiles_config(filename_path):

  system_prompt =""
  anchor_schema ={}
  anchorlist_scehma={}
  empty_results_schema={}

  with open(filename_path, encoding='utf-8') as file:#always use encoding="utf-8"(or corresponding encoding) to avoid errors
      content = file.read()

      content_list = re.split("\\n[=]+", content)
      #content_list = re.split(r"(?<=\\n[=])", content)

      for elm in content_list:
          if(not elm):
              continue

          elm =re.sub(r'[=]+','', elm)


          if(re.search("^PROMPT SCHEMA", elm)):
              system_prompt = elm.replace("PROMPT SCHEMA","").strip()


          if(re.search("^ANCHOR/QUESTIONS TYPES", elm)):
              chunk_question_json = elm.replace("ANCHOR/QUESTIONS TYPES","").strip()
              anchorlist_schema = json.loads(chunk_question_json)

          if(re.search("^ANCHOR SCHEMA", elm)):
              elm_json = elm.replace("ANCHOR SCHEMA","").strip()
              anchor_schema = json.loads(elm_json)

          if(re.search("^EMPTY_RESULTS", elm)):
              elm_json = elm.replace("EMPTY_RESULTS","").strip()
              empty_results_schema = json.loads(elm_json)

      return(system_prompt, anchorlist_schema, anchor_schema, empty_results_schema)

Here we alter the values :- **None** and **False** back to original form.  These were not **string** but enteredas strings in schema.txt file to avoid parsing error while reading the file

In [ ]:
def modify_dict_var(input_json, json_type):
    if(json_type =='anchor_schema'):
        for key in input_json.keys():
            input_json[key]['additionalProperties'] = False

    if(json_type =='empty_results_schema'):
        for key in input_json.keys():
            for key2 in input_json[key].keys():
                input_json[key][key2] = None

    return input_json

**need to switchlinks here for old/new initial/prep data**

In [ ]:
schema_filename_path =path+old_data_folder+ "/old_data_schema_iter1.txt"
#schema_filename_path =path+new_data_folder + "/new_data_schema_iter2.txt"

In [ ]:
SYSTEM_PROMPT, CHUNK_QUESTION_SCHEMA, ANCHOR_SCHEMA, EMPTY_RESULTS_SCHEMA = readfiles_config(schema_filename_path)

ANCHOR_SCHEMA, EMPTY_RESULTS_SCHEMA = modify_dict_var(ANCHOR_SCHEMA, 'anchor_schema'), modify_dict_var(EMPTY_RESULTS_SCHEMA, 'empty_results_schema')

#print(SYSTEM_PROMPT)
print('-'*50)
pprint(CHUNK_QUESTION_SCHEMA)
print('-'*50)
pprint(ANCHOR_SCHEMA)
print('-'*50)
pprint(EMPTY_RESULTS_SCHEMA)

--------------------------------------------------
{'Positive_chunks': ['macro_question',
                     'patient_profile_question',
                     'operational_question',
                     'conversational_question']}
--------------------------------------------------
{'Positive_chunks': {'additionalProperties': False,
                     'properties': {'conversational_question': {'type': 'string'},
                                    'macro_question': {'type': 'string'},
                                    'operational_question': {'type': 'string'},
                                    'patient_profile_question': {'type': 'string'}},
                     'required': ['macro_question',
                                  'patient_profile_question',
                                  'operational_question',
                                  'conversational_question'],
                     'type': 'object'}}
--------------------------------------------------
{'Positive_chunks

In [ ]:
!pip install ollama

**Blocks to ensure ollama is running**

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,669 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 https://

In [ ]:
!sudo apt update && sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
76 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' 

In [ ]:
!nohup ollama serve > ollama.log 2>&1 & #to start ollama server

In [ ]:
# Verify the background service is active
!curl http://127.0.0.1:11434

Ollama is running

**LLM setup over ollama server(max_workers, keep_alive), load it**

In [ ]:
# ============================================================
# 2. OLLAMA CONFIGURATION
# ============================================================

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
MODEL = "llama3.1:8b-instruct-q4_K_M"

NUM_RECORDS = 2000

# Start with 2 on a T4.
# Increase to 3 only after checking VRAM / throughput.
MAX_WORKERS = 4

REQUEST_TIMEOUT = 120

# Keep model loaded in VRAM(keeplive ="30m", "10m", -1)
KEEP_ALIVE = -1

In [ ]:
'''
# Enable GPU support, turn off unneeded features, and use all  CPU cores(-1) available on free Colab(else model download in ollama model would take a lot of time)
%env CMAKE_ARGS=-DGGML_CUDA=on -DLLAVA_BUILD=off
%env CMAKE_BUILD_PARALLEL_LEVEL=-1
'''
#1.Pull model:
!ollama pull llama3.1:8b-instruct-q4_K_M

#!ollama pull llama3.1:8b-instruct-q4_K_M

In [ ]:
# 2. Set your concurrency parameters (affects runtime speed, not download speed)
os.environ["OLLAMA_NUM_PARALLEL"] = f"{MAX_WORKERS}"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

**prompt creation (both system instructon and user prompt), define fns for anchor extraction, initialize payload parameters followed by api hit to ollama url and extract in parallel**

Although:- System_queries :- Permanent rules+ quality constraints(static)+ User_queries:- Qestion definitions + Trial(individual positive_chunk)===>we have enlarged system_query and shrinked use_query(largely only retreival chunk).

**Reason:-** Ollama’s /api/generate endpoint actually accepts a dedicated "system" field directly in its root payload schema, allowing you to keep them perfectly segregated and tokeizes them once. In sucessive iters on only user_prompt(dynamic_trial chunk) gets tokenized while system_prompt tokens are fed from GPU's VRAM

In [ ]:
# ============================================================
# 3. PROMPT
# ============================================================
SYSTEM_INSTRUCTIONS = '\n'+SYSTEM_PROMPT+'\n'


In [ ]:
#print(SYSTEM_INSTRUCTIONS)

Prompt building and structure validation

In [ ]:
def build_prompt(chunk_type, positive_chunk):
  return f"""
{SYSTEM_INSTRUCTIONS}

chunk_type: {chunk_type}

CLINICAL TRIAL TEXT:

{positive_chunk}

Generate the required anchor question(s) now.
"""

#JSON cleaning/validation:-

def validate_result(chunk_type, obj):
    REQUIRED_KEYS= CHUNK_QUESTION_SCHEMA[chunk_type]#REQUIRED_KEYS already read from schema file above

    if not isinstance(obj, dict):
        return False

    if set(REQUIRED_KEYS) != set(obj.keys()):
        return False

    for key in REQUIRED_KEYS:

        value = obj[key]

        if not isinstance(value, str):
            return False

        value = value.strip()

        if len(value) < 10:
            return False

        # Must be interrogative
        if not value.endswith("?"):
            return False

    # Ensure questions are not identical
    values = [obj[k].strip().lower() for k in REQUIRED_KEYS]

    if len(set(values)) != len(REQUIRED_KEYS):
        return False

    return True

chunk level response generation.

In [ ]:
def generate_anchors(record_id, chunk_type, positive_chunk, max_retries=3):
    CHUNK_ANCHOR_SCHEMA = ANCHOR_SCHEMA[chunk_type]
    #CHUNK_EMPTY_RESULTS_SCHEMA = EMPTY_RESULTS_SCHEMA[chunk_type]

    prompt = build_prompt(chunk_type, positive_chunk)

    payload = {
    "model": MODEL,
    "prompt": prompt,
    "stream": False,
    "keep_alive": KEEP_ALIVE,

    "format": CHUNK_ANCHOR_SCHEMA,

    "options": {
        "temperature": 0.1,#def=0.4
        "top_p": 0.9,
        "top_k": 30,#def=40
        "repeat_penalty": 1.05,
        "num_predict": 200,
        "num_ctx": 8192
    }}

    last_error = None

    for attempt in range(max_retries):

        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()

            data = response.json()
            raw_output = data["response"]
            result = json.loads(raw_output)

            if validate_result(chunk_type, result):

                return {
                    "record_id": record_id,
                    "chunk_type": chunk_type,
                    **result,
                    "status": "success"
                }

            raise ValueError(
                "Generated JSON failed validation"
            )

        except Exception as e:

            last_error = str(e)
            #print(f"Error: {last_error}")

            # Exponential retry
            time.sleep(2 ** attempt)

    return {
    "record_id": record_id,
    "chunk_type": chunk_type,
    **EMPTY_RESULTS_SCHEMA[chunk_type],
    "status": f"failed: {last_error}"
    }


druing parallel generation, need to be alert to **either feed row['chunk_type'](for new data) or "Positive_chunks"(for old data)** while generating record(list of tuples:- nctId, chunk_type, chunks)

In [ ]:
from requests.models import ChunkedEncodingError
# ============================================================
# 6. PARALLEL GENERATION
# ============================================================

def generate_all(df):

    records = [
        (
            row["nctId"],

            "Positive_chunks",#in case of old initial data since no chunk_type field(commeneted in case of old data)
            #row["type"],#in case of new initial data(commeneted in case of old data)

            row["Positive_chunks"],##in case of old initial data(commented in case of new)
            #row["chunk"]#in case of new initial data(commeneted in case of old data)
        )
        for idx, row in df.iterrows()
    ]

    results = []

    start_time = time.time()

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {executor.submit(generate_anchors, record_id, chunk_type, chunk): (record_id,chunk_type)
                   for record_id, chunk_type, chunk in records}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Generating anchors"):

            try:
                result = future.result()
                results.append(result)

            except Exception as e:#although already handled in generate_anchors exception segment and last return(just a defensive formality)
                record_id = futures[future][0]
                chunk_type = futures[future][1]

                results.append({
                    "record_id": record_id,
                    "chunk_type": chunk_type,
                    "status": f"failed: {e}"
                })

    elapsed = time.time() - start_time

    print(
        f"\nCompleted {len(results):,} records "
        f"in {elapsed / 3600:.2f} hours"
    )

    return pd.DataFrame(results)




Run genrerate_all for raw df on extracted anchors

In [ ]:
import time

start = time.time()

#test_results = generate_all(df_positive_chunk_old.head(10))##---->for or old/new:- generate_all(df_positive_chunk_(old/new).head(10))
generated_df = generate_all(df_positive_chunk_old)#---->for or old/new:- generate_all(df_positive_chunk_(old/new))

elapsed = time.time() - start

print(f"6000 records: {elapsed:.1f} seconds")
print(f"Per record: {elapsed / 20:.2f} sec")
print(
    f"Estimated 2000 records: "
    f"{elapsed / 20 * 2000 / 3600:.2f} hours"
)

Generating anchors:   0%|          | 0/2000 [00:00<?, ?it/s]


Completed 2,000 records in 2.13 hours
6000 records: 7686.0 seconds
Per record: 384.30 sec
Estimated 2000 records: 213.50 hours


In [ ]:
#test_results.head(20)
generated_df.head()

,record_id,chunk_type,macro_question,patient_profile_question,operational_question,conversational_question,status
0,NCT07640360,Positive_chunks,What is the main goal of this clinical trial r...,Could a patient with a recent minor ischemic s...,Is WiFi access required for patients participa...,"I had a small stroke, can I use a smartwatch t...",success
1,NCT07690982,Positive_chunks,What is the purpose of this clinical trial reg...,Could a patient with hemiplegia caused solely ...,How old must a patient be to participate in th...,I had a stroke and I'm interested in using an ...,success
2,NCT07635498,Positive_chunks,What is the main goal of this clinical trial c...,Could a patient with acute ischemic stroke who...,How old must a patient be to participate in th...,I had a stroke and can't get to a special stro...,success
3,NCT07631078,Positive_chunks,What is the primary mechanism by which kTMP-en...,Could a patient who had an ischemic stroke 15 ...,Is this clinical trial open to participants fr...,I had a stroke a year ago and still have troub...,success
4,NCT07702968,Positive_chunks,What is the main goal of this clinical trial r...,Could a patient with symptoms of acute stroke ...,Is there an age restriction for patients parti...,"I'm having a stroke, can I get a special MRI s...",success
5,NCT07629674,Positive_chunks,What is the primary goal of this clinical tria...,Could a patient with a subacute post-stroke he...,How old must a participant be to be eligible f...,I had a stroke and I'm still having trouble wa...,success
6,NCT07753759,Positive_chunks,What is being compared in this clinical trial?,Could I qualify for this trial if I have post-...,How old do you need to be to participate in th...,I had a stroke and now I'm experiencing should...,success
7,NCT07670585,Positive_chunks,What is the primary objective of this clinical...,Could a patient with a history of ischemic str...,Is there an age restriction for participating ...,"I had a stroke, can I participate in this stud...",success
8,NCT07747701,Positive_chunks,What is the main goal of this clinical trial i...,Could a patient with an NIHSS score of 8 and a...,Is this clinical trial open to patients aged 7...,"I had a stroke last week, can I participate in...",success
9,NCT07647666,Positive_chunks,What is the purpose of combining an oropharyng...,Could a patient with impaired consciousness an...,Is enrollment in this trial limited to patient...,I had a stroke and now I have pneumonia. Can I...,success


In [ ]:
generated_df.to_parquet(path+old_data_folder+"/4anchor_+veCleansed_CT.parquet")#folder paths:-path+old_data_folder-->for old; path+new_data_folder--->new

Duplication check with csoine sim alongside normalization

In [ ]:
generated_df = pd.read_parquet(path+old_data_folder+"/4anchor_+veCleansed_CT.parquet")#folder paths:-path+old_data_folder-->for old; path+new_data_folder--->new

In [ ]:
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Define normalization function, generate extracted anchors from LLM, initialize embedding model and then excute normalizaton(alonside with cosine similarity)**

In [ ]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

In [ ]:
# ============================================================
# NORMALIZE + COSINE-BASED ANCHOR FILTERING
# ============================================================

def normalize_anchor_results(
    generated_df,
    initial_df,
    iteration="new",
    similarity_threshold=0.85
):

    if iteration == "old":

        anchor_cols = [
            "macro_question",
            "patient_profile_question",
            "operational_question",
            "conversational_question"
        ]

        meta = initial_df[
            ["nctId", "Positive_chunks", "char_len"]
        ].rename(columns={
            "nctId": "record_id",
            "Positive_chunks": "positive",
            "char_len": "chunk_char_length"
        })

        #meta["chunk_type"] = "consolidated"#already populated in "generate_all fn"
        meta["block_no"] = 1

        merged = generated_df.merge(
            meta,
            on="record_id",
            how="left",
            validate="one_to_one"
        )

    elif iteration == "new":

        anchor_cols = [
            "trial_topic_question",
            "study_objective_question",
            "clinical_population_question",
            "patient_profile_question",
            "eligibility_constraint_question"
        ]

        meta = initial_df[
            ["nctId", "type", "chunk", "chunk_length", "block_no"]
        ].rename(columns={
            "nctId": "record_id",
            "type": "chunk_type",
            "chunk": "positive",
            "chunk_length": "chunk_char_length"
        })

        merged = generated_df.merge(
            meta,
            on=["record_id", "chunk_type"],
            how="left",
            validate="one_to_one"
        )

    else:
        raise ValueError("iteration must be 'old' or 'new'")

    #return(merged)#added for validation

    # --------------------------------------------------------
    # Wide → long
    # --------------------------------------------------------


    rows = []

    for _, r in merged.iterrows():

        for anchor_type in anchor_cols:

            anchor = r.get(anchor_type)

            if pd.isna(anchor):
                continue

            rows.append({
                "nctId": r["record_id"],
                "chunk_type": r["chunk_type"],
                "block_no": r["block_no"],
                "document_id": f'{r["record_id"]}_{int(r["block_no"])}',
                "anchor_type": anchor_type,
                "anchor": (
                    str(anchor).strip()
                    if pd.notna(anchor) else None
                ),
                "positive": r["positive"],
                "chunk_char_length": r["chunk_char_length"],
                "status": r.get("status", "failed")
            })

    normalized_df = pd.DataFrame(rows)


    # --------------------------------------------------------
    # Cosine filtering
    # --------------------------------------------------------


    rejection_set = set()

    for nctid, group in normalized_df.groupby(
        "nctId", sort=False
    ):

        valid = group[
            (group["status"] == "success") &
            group["anchor"].notna()
        ]

        if len(valid) <= 1:
            continue

        df_indices = valid.index.tolist()
        anchors = valid["anchor"].tolist()

        embeddings = embed_model.encode(
            anchors,
            show_progress_bar=False
        )

        local_rejection = set()

        for i in range(len(anchors)):

            if i in local_rejection:
                continue

            for j in range(i + 1, len(anchors)):

                if j in local_rejection:
                    continue

                if cosine_similarity(
                    embeddings[i],
                    embeddings[j]
                ) > similarity_threshold:

                    # Reject second anchor
                    local_rejection.add(j)

        rejection_set.update(
            df_indices[i] for i in local_rejection
        )

    # --------------------------------------------------------
    # Remove rejected anchors
    # --------------------------------------------------------

    normalized_df = (
        normalized_df
        .drop(index=rejection_set)
        .reset_index(drop=True)
    )

    return normalized_df


very important to **change the df_positive_chun_old/new and iteration** as commented beside code

In [ ]:
normalized_df = normalize_anchor_results(
    #test_results,
    generated_df,

    df_positive_chunk_old,#----->old:-df_positive_chunk_old, new:- df_positive_chunk_old
    iteration="old",#---->old:- old, new:- new

    similarity_threshold=0.85#default=0.90
)

In [ ]:
normalized_df.head(10)

,nctId,chunk_type,block_no,document_id,anchor_type,anchor,positive,chunk_char_length,status
0,NCT07640360,Positive_chunks,1,NCT07640360_1,macro_question,What is the main goal of this clinical trial r...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
1,NCT07640360,Positive_chunks,1,NCT07640360_1,patient_profile_question,Could I qualify for this trial if I had a rece...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
2,NCT07640360,Positive_chunks,1,NCT07640360_1,operational_question,Is there an age limit for patients to particip...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
3,NCT07640360,Positive_chunks,1,NCT07640360_1,conversational_question,I had a small stroke and my doctor said I shou...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
4,NCT07631078,Positive_chunks,1,NCT07631078_1,macro_question,What is the primary mechanism by which kTMP-en...,TITLE: kTMP-Enhanced Motor Rehabilitation for ...,750,success
5,NCT07631078,Positive_chunks,1,NCT07631078_1,patient_profile_question,Could a patient who had an ischemic stroke 15 ...,TITLE: kTMP-Enhanced Motor Rehabilitation for ...,750,success
6,NCT07631078,Positive_chunks,1,NCT07631078_1,operational_question,What is the minimum age requirement for partic...,TITLE: kTMP-Enhanced Motor Rehabilitation for ...,750,success
7,NCT07631078,Positive_chunks,1,NCT07631078_1,conversational_question,I had a stroke a year ago and still have troub...,TITLE: kTMP-Enhanced Motor Rehabilitation for ...,750,success
8,NCT07690982,Positive_chunks,1,NCT07690982_1,macro_question,What is the purpose of this clinical trial reg...,TITLE: Validity and Reliability of the PhysioM...,1839,success
9,NCT07690982,Positive_chunks,1,NCT07690982_1,patient_profile_question,Could a patient with hemiplegia caused solely ...,TITLE: Validity and Reliability of the PhysioM...,1839,success


In [2]:
#normalized_df.iloc[8].to_dict()

In [ ]:
normalized_df = normalized_df.replace('', np.nan).dropna()

In [ ]:
#reorganize scattered nctId rows together
normalized_final_list=[]
for nctid, group in normalized_df.groupby("nctId", sort=False):
    normalized_final_list.append(group)

normalized_final_df = pd.concat(normalized_final_list)

#reset index after reshuffle
normalized_final_df =normalized_final_df.reset_index(drop =True)

#insert anchor_id at 0th index:-
normalized_final_df.insert(0, "anchor_id",normalized_final_df.index)

In [ ]:
normalized_final_df['status'].unique()

array(['success'], dtype=object)

**Finding duplicate chunks(mostly inclusion criteria that fall in more than one CT(ncTIds))---> common to both old and new data**

In [ ]:
print(f"shpae of normalized df:-{normalized_final_df.shape}; unique_anchors:-{normalized_final_df['anchor'].nunique()}; unique_status:-{normalized_final_df['status'].unique()}"
f"\nunique_doc_ids:{normalized_final_df['document_id'].nunique()};unique_positives/doc:{normalized_final_df["positive"].nunique()} ")

shpae of normalized df:-(7966, 10); unique_anchors:-6890; unique_status:-['success']
unique_doc_ids:2000;unique_positives/doc:1999 


In [ ]:
duplicate_chunk=[]

positive_list = normalized_final_df["positive"].unique().tolist()
for positive in positive_list:
    df_positive =normalized_final_df[normalized_final_df["positive"]==positive]
    if(df_positive.shape[0]>4):
        duplicate_chunk.append(df_positive.index.tolist())

print(f"duplicate chunks(inclusion_criteria specifically) across multiple nctId\n{duplicate_chunk}")


duplicate chunks(inclusion_criteria specifically) across multiple nctId
[[1371, 1372, 1373, 1374, 1375, 1376, 1377, 1378]]


In [ ]:
normalized_final_df.iloc[duplicate_chunk[0]]

,anchor_id,nctId,chunk_type,block_no,document_id,anchor_type,anchor,positive,chunk_char_length,status
1371,1371,NCT07666971,Positive_chunks,1,NCT07666971_1,macro_question,What is the main goal of this clinical trial r...,TITLE: Prevention of Rebound Pain After Orthop...,1103,success
1372,1372,NCT07666971,Positive_chunks,1,NCT07666971_1,patient_profile_question,Could a patient with upper limb orthopaedic su...,TITLE: Prevention of Rebound Pain After Orthop...,1103,success
1373,1373,NCT07666971,Positive_chunks,1,NCT07666971_1,operational_question,Is this clinical trial open to patients aged o...,TITLE: Prevention of Rebound Pain After Orthop...,1103,success
1374,1374,NCT07666971,Positive_chunks,1,NCT07666971_1,conversational_question,"I'm having surgery on my arm, will I get extra...",TITLE: Prevention of Rebound Pain After Orthop...,1103,success
1375,1375,NCT07666009,Positive_chunks,1,NCT07666009_1,macro_question,What is the primary goal of this clinical tria...,TITLE: Prevention of Rebound Pain After Orthop...,1103,success
1376,1376,NCT07666009,Positive_chunks,1,NCT07666009_1,patient_profile_question,Could a patient with upper limb orthopaedic su...,TITLE: Prevention of Rebound Pain After Orthop...,1103,success
1377,1377,NCT07666009,Positive_chunks,1,NCT07666009_1,operational_question,Is this trial limited to patients aged over 18...,TITLE: Prevention of Rebound Pain After Orthop...,1103,success
1378,1378,NCT07666009,Positive_chunks,1,NCT07666009_1,conversational_question,"I'm having surgery on my arm, will I get extra...",TITLE: Prevention of Rebound Pain After Orthop...,1103,success


**DEbuggin_ends(not to be done in old data_type:-4synthetic anchors to 1 chunk format)**

In order to fix these just document_id(nctId+ block_no) wont suffice. We'll have to create hashes (and use them as ***unique_chunk_identifiers*** as actual index of chunk) out of chunks to ensure optimum ***content-level retrieval*** from any anchors. For CT mapoing afterwards, we are also preserving ***document_id***. Also priortizing (content-level retrieval> clinical trial record retreival) helps:-
- Wider application :- anchor-to-anchor, anchor-to-chunk, chunk-to-chunk simlarity analysis.
- Better precision, NDCG, MRR metrics etc in inference (opposite to when document_id would have been used as it would have penalized the anchor-positive(indices mapping) if anchor showed correlation with same chunk in different CT(ncTid).
- The fact that ***unique_chunk_identifiers & document_id*** coexist shall  aid a local anchor ~$a_i$~ (specific to a CT nctId) of same chunk ~$c_i$~ (present in different nctIds) to align with all instances of the chunk ~$C_I$~= [~$c_i1$~,~$c_i2$~,...~$c_iN$~]

In [ ]:
def add_unique_chunk_identifier(df):

    df = df.copy()

    df["unique_chunk_identifier"] = df["positive"].apply(
        lambda x: hashlib.sha256(
            re.sub(r"\s+", " ", str(x).strip()).encode("utf-8")
        ).hexdigest()
        if pd.notna(x) and str(x).strip()
        else None
    )

    col = df.pop('unique_chunk_identifier')
    df.insert(9, 'unique_chunk_identifier', col)


    return df

In [ ]:
normalized_final_df = add_unique_chunk_identifier(normalized_final_df)

In [6]:
normalized_final_df.head()

,anchor_id,nctId,chunk_type,block_no,document_id,anchor_type,anchor,positive,chunk_char_length,status
0,0,NCT07640360,Positive_chunks,1,NCT07640360_1,macro_question,What is the main goal of this clinical trial r...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
1,1,NCT07640360,Positive_chunks,1,NCT07640360_1,patient_profile_question,Could I qualify for this trial if I had a rece...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
2,2,NCT07640360,Positive_chunks,1,NCT07640360_1,operational_question,Is there an age limit for patients to particip...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
3,3,NCT07640360,Positive_chunks,1,NCT07640360_1,conversational_question,I had a small stroke and my doctor said I shou...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
4,4,NCT07631078,Positive_chunks,1,NCT07631078_1,macro_question,What is the primary mechanism by which kTMP-en...,TITLE: kTMP-Enhanced Motor Rehabilitation for ...,750,success


In [ ]:
normalized_final_df.shape

(7866, 10)

In [ ]:
normalized_final_df.to_parquet(path+old_data_folder+"/normalized_anchor-positive_pairsCleansedOld_CT.parquet")

**Push data to huggingface hub**

We'll ***(i)authenticate in colab (by adding generated tokens to colab's secrets)*** rather than ***(ii)interative sign_in to colab(login,logout)*** as the former as its more safe and secure. Steps for implementing it:-

1.  Copy the universal **HF_TOKEN**(generated by access tokens drop_down on profile logo on top right of huggingface account) up in secret side_panel of colab after add new key section.
2.  while pasting the value should **name and value** fields must have **'HF_TOKEN' and 'token_id' (generated in huggingface)**

3.  Since we are reading(here) and writing(when pushed data previuosly data):- access_key creation must have **preset:-Write(covers both read and write)**

4.  Injcet fine grained token into environment's current variable(although may be ignored as token is ingrained by default in colab)

5.  If one wants verification of read and write to from and to hub, invoke HfApi below and follow syntax in 2nd next block

In [5]:
#normalized_final_df = pd.read_parquet(path+old_data_folder+"/normalized_anchor-positive_pairsCleansedOld_CT.parquet")

In [7]:
# 1. Inject the fine-grained token into your current environment variables
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
api = HfApi()

In [8]:
user_info = api.whoami()
username = user_info['name']

dataset = Dataset.from_pandas(normalized_final_df)

dataset.push_to_hub(f"{username}/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data_final")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  26%|##6       |  537kB / 2.03MB            

CommitInfo(commit_url='https://huggingface.co/datasets/vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data_final/commit/8d13bed841f5112e508553e9e3a3201f6823e48a', commit_message='Upload dataset', commit_description='', oid='8d13bed841f5112e508553e9e3a3201f6823e48a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data_final', endpoint='https://huggingface.co', repo_type='dataset', repo_id='vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data_final'), pr_revision=None, pr_num=None)

In [ ]:
#!killall drive